In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import permutations
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as SklearnRF

In [ ]:
def gini_impurity(y):
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1.0 - np.sum(probs ** 2)


def best_split(X, y, n_features):
    n, d = X.shape
    best_gain = -np.inf
    best_feat, best_thresh = None, None
    parent_gini = gini_impurity(y)

    feat_idx = np.random.choice(d, min(n_features, d), replace=False)
    for feat in feat_idx:
        thresholds = np.unique(X[:, feat])
        for thresh in thresholds:
            left = y[X[:, feat] <= thresh]
            right = y[X[:, feat] > thresh]
            if len(left) == 0 or len(right) == 0:
                continue
            gain = parent_gini - (len(left)/n)*gini_impurity(left) - (len(right)/n)*gini_impurity(right)
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, feat, thresh

    return best_feat, best_thresh


print('Gini / split helpers defined.')

In [ ]:
class _Node:
    __slots__ = ('feat', 'thresh', 'left', 'right', 'value')
    def __init__(self, feat=None, thresh=None, left=None, right=None, value=None):
        self.feat, self.thresh, self.left, self.right, self.value = feat, thresh, left, right, value


class DecisionTree:
    def __init__(self, max_depth=10, min_samples=2, n_features=None):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        nf = self.n_features or X.shape[1]
        self.root = self._grow(X, y, 0, nf)
        return self

    def _grow(self, X, y, depth, nf):
        if depth >= self.max_depth or len(y) < self.min_samples or len(np.unique(y)) == 1:
            return _Node(value=int(np.bincount(y).argmax()))
        feat, thresh = best_split(X, y, nf)
        if feat is None:
            return _Node(value=int(np.bincount(y).argmax()))
        mask = X[:, feat] <= thresh
        return _Node(
            feat=feat, thresh=thresh,
            left=self._grow(X[mask],  y[mask],  depth+1, nf),
            right=self._grow(X[~mask], y[~mask], depth+1, nf),
        )

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.value is not None:
            return node.value
        return self._traverse(x, node.left if x[node.feat] <= node.thresh else node.right)


print('DecisionTree class defined.')

In [ ]:
class RandomForest:
    def __init__(self, n_trees=100, max_depth=10, min_samples=2, n_features='sqrt'):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.n_features = n_features
        self.trees_ = []
        self.oob_score_ = None

    def _resolve(self, d):
        if self.n_features == 'sqrt':  return max(1, int(np.sqrt(d)))
        if self.n_features == 'log2':  return max(1, int(np.log2(d)))
        return self.n_features

    def fit(self, X, y):
        n, d = X.shape
        nf = self._resolve(d)
        n_classes = len(np.unique(y))
        self.trees_ = []
        oob_votes  = np.zeros((n, n_classes), dtype=int)
        oob_counts = np.zeros(n, dtype=int)

        for _ in range(self.n_trees):
            idx = np.random.choice(n, n, replace=True)
            oob = np.setdiff1d(np.arange(n), np.unique(idx))
            tree = DecisionTree(max_depth=self.max_depth, min_samples=self.min_samples, n_features=nf)
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)
            if len(oob):
                for i, pred in zip(oob, tree.predict(X[oob])):
                    oob_votes[i, pred] += 1
                    oob_counts[i] += 1

        valid = oob_counts > 0
        if valid.sum() > 0:
            self.oob_score_ = np.mean(np.argmax(oob_votes[valid], axis=1) == y[valid])
        return self

    def predict(self, X):
        votes = np.array([t.predict(X) for t in self.trees_])  # (n_trees, n_samples)
        return np.array([np.bincount(col).argmax() for col in votes.T])

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

    def feature_importances(self, X, y):
        """Permutation importance: mean accuracy drop when each feature is shuffled."""
        baseline = self.accuracy(X, y)
        imps = []
        for f in range(X.shape[1]):
            Xp = X.copy()
            np.random.shuffle(Xp[:, f])
            imps.append(baseline - self.accuracy(Xp, y))
        total = sum(max(0.0, v) for v in imps)
        return np.array([max(0.0, v) / total if total else 0.0 for v in imps])


print('RandomForest class defined.')

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForest(n_trees=50, max_depth=8)
rf.fit(X_train, y_train)

print(f'Train Accuracy : {rf.accuracy(X_train, y_train):.4f}')
print(f'Test  Accuracy : {rf.accuracy(X_test,  y_test):.4f}')
print(f'OOB   Score    : {rf.oob_score_:.4f}')

In [ ]:
print(f'{"n_trees":>8} | {"Train":>8} | {"Test":>8}')
print('-' * 30)
tree_counts = [1, 5, 10, 20, 50, 100]
test_accs = []
for n in tree_counts:
    rf_n = RandomForest(n_trees=n, max_depth=8)
    rf_n.fit(X_train, y_train)
    ta = rf_n.accuracy(X_train, y_train)
    va = rf_n.accuracy(X_test, y_test)
    test_accs.append(va)
    print(f'{n:>8} | {ta:>8.4f} | {va:>8.4f}')

plt.figure(figsize=(7, 4))
plt.plot(tree_counts, test_accs, marker='o')
plt.xlabel('Number of Trees'); plt.ylabel('Test Accuracy')
plt.title('Random Forest - Effect of n_trees')
plt.xticks(tree_counts)
plt.tight_layout(); plt.show()

In [ ]:
importances = rf.feature_importances(X_test, y_test)
feat_names  = cancer.feature_names
idx = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), importances[idx], color='steelblue', edgecolor='black')
plt.xticks(range(10), feat_names[idx], rotation=40, ha='right')
plt.title('Top 10 Feature Importances (Permutation)')
plt.ylabel('Normalised Accuracy Drop')
plt.tight_layout(); plt.show()

In [ ]:
sk_rf = SklearnRF(n_estimators=50, max_depth=8, random_state=42)
sk_rf.fit(X_train, y_train)

print(f'sklearn RandomForest Test Accuracy : {sk_rf.score(X_test, y_test):.4f}')
print(f'From-scratch Test Accuracy         : {rf.accuracy(X_test, y_test):.4f}')

In [ ]:
# Multi-class: Wine dataset (3 classes)
wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=42)

rf_wine = RandomForest(n_trees=50, max_depth=6)
rf_wine.fit(Xw_tr, yw_tr)

print(f'Wine dataset (3-class)')
print(f'Train Accuracy : {rf_wine.accuracy(Xw_tr, yw_tr):.4f}')
print(f'Test  Accuracy : {rf_wine.accuracy(Xw_te, yw_te):.4f}')
print(f'OOB   Score    : {rf_wine.oob_score_:.4f}')